# Supervised ML PipeLine Template

This notebook contains a comprehensive ML pipeline template for the following model:
2) XGBoost

Additonally, the follow undersample methods can be chosen:
1) RandomUnderSampler

### Some of the assumptions: 
1) Data is pre-cleaned with no missing values.
2) Only numeric features are used as all other columns are excluded automatically.

## How will this be used (steps):
1) Only run in new notebooks, import the module as follows: **from Supervised_ML_Pipeline import MLPipeline**
2) Declare an object and initialize it as follows: **pipe = MLPipeline(model_name="svm", target_col="outcome")**
3) This line does the data spliting, training & testing and provides results: **pipe.run(df, param_grid={}, group_col="", exclude_cols=[], undersample_method=None)**

Important - Read the below parameters

#### Required parameters:
1) model_name - can't run without knowing which model
2) target_col - can't extract features without knowing the target
3) param_grid - can't run GridSearchCV without parameters
4) group_col - can't do StratifiedGroupKFold without groups

#### Optional parameters:
1) exclude_cols - defaults to [] if not passed, pipeline still runs fine
2) undersample_method - defaults to None, no undersampling applied
3) output_dir - defaults to ".", saves in current directory
4) cv, scoring, test_size, random_state, verbose - all have sensible defaults (5, average_precision, 0.4, 42, 1)

#### Additional stuff:
1) list_models() gives a list of models available
2) list_undersample_methods() gives a list of undersampling methods available

## Results:
1) Displays a quick summary of the training along with evaluation metrics conducted on test data.
2) Saves a score distrution plot and confusion matrix plot as png.
3) Saves a csv file with test scores and their respective risk label.
4) Saves a summary, waterfall and feature importance plot as png (SHAP).


In [1]:
# In the Libraries portion of a new different notebook
from Supervised_ML_Pipeline import MLPipeline, add_inne_features
import pandas as pd

In [2]:
MLPipeline.list_models()

['logistic_regression', 'xgboost', 'svm']

In [3]:
MLPipeline.list_undersample_methods()

['RandomUnderSampler', 'CLEANSE']

In [4]:
MLPipeline.list_feature_selection_methods()

['k_best_catboost', 'k_best_xgb', 'k_best_lgbm', 'k_best_decision_tree']

In [5]:
select_methods = MLPipeline.list_feature_selection_methods()
select_methods

['k_best_catboost', 'k_best_xgb', 'k_best_lgbm', 'k_best_decision_tree']

In [6]:
df = pd.read_csv("/dsa/groups/casestudycf25/team02/silver/unified_dme_year_npi_hcpcs_rentl_ind_labeled.csv")

target_col = 'target'

exclude_cols = [
    'npi', 'rfrg_prvdr_state_abrvtn',
    'specialty_type', 'specialty'
]

df.head(5)

,npi,hcpcs_cd,suplr_rentl_ind,tot_suplrs,tot_suplr_benes,tot_suplr_clms,tot_suplr_srvcs,avg_suplr_sbmtd_chrg,avg_suplr_mdcr_alowd_amt,year,...,claims_per_bene_zscore_by_type,claims_per_bene_outlier_by_type,services_per_bene_zscore,services_per_bene_outlier,services_per_bene_zscore_by_type,services_per_bene_outlier_by_type,benes_per_supplier_zscore,benes_per_supplier_outlier,benes_per_supplier_zscore_by_type,benes_per_supplier_outlier_by_type
0,1003000902,A4253,0,11,14.0,32,67,68.483134,8.386567,2021,...,0.0,False,0.0,False,0.0,False,0.0,False,0.0,False
1,1003000902,A4259,0,8,5.0,14,17,14.727647,1.437647,2021,...,0.0,False,0.0,False,0.0,False,0.0,False,0.0,False
2,1003000902,A4390,0,3,5.0,17,330,16.484364,11.178182,2021,...,0.0,False,0.0,False,0.0,False,0.0,False,0.0,False
3,1003000902,E0570,1,1,5.0,12,12,50.730000,5.921667,2021,...,0.0,False,0.0,False,0.0,False,0.0,False,0.0,False
4,1003000902,E1390,1,2,5.0,24,24,389.755000,111.531667,2021,...,0.0,False,0.0,False,0.0,False,0.0,False,0.0,False


In [7]:
df.columns.to_list()

['npi',
 'hcpcs_cd',
 'suplr_rentl_ind',
 'tot_suplrs',
 'tot_suplr_benes',
 'tot_suplr_clms',
 'tot_suplr_srvcs',
 'avg_suplr_sbmtd_chrg',
 'avg_suplr_mdcr_alowd_amt',
 'year',
 'tot_suplr_mdcr_pymt_amt',
 'tot_suplr_mdcr_pymt_amt_hcpcs_rentl_ind_z_scr',
 'avg_suplr_mdcr_alowd_amt_hcpcs_rentl_ind_rat',
 'target',
 'n_manufacturing_entities',
 'total_payment',
 'third_party_equals_covered_recipient_indicator',
 'covered_biological',
 'covered_device',
 'covered_drug',
 'covered_medical supply',
 'non-covered_device',
 'non-covered_drug',
 'non-covered_medical supply',
 'is_weekend_payment',
 'is_q4_payment',
 'dme_urological_supplies',
 'dme_surgical_dressings',
 'dme_lower_limb_orthotics',
 'dme_parenteral_nutrition',
 'dme_ventilators',
 'dme_oxygen_accessories',
 'dme_oxygen_equipment',
 'dme_humidifiers_and_nebulizers',
 'dme_wheelchair_access',
 'dme_pharmacy_dispensing',
 'dme_infusion_pumps',
 'dme_breathing_aids',
 'dme_hospital_beds',
 'dme_replacement_batteries',
 'dme_tapes_

In [8]:
param_grid = {
    "model__C": [0.001, 0.1, 1, 10],
    "model__penalty": ["l1", "l2", "elasticnet"],
    "model__solver": ["saga"],
    "model__l1_ratio": [0.5, 0, 1.0],
    "model__max_iter": [500],
    "model__class_weight": ["balanced", {1: 250}, {1: 1500}],
    "selector__k": [25]
}

In [9]:
###########################
# RUN HISTORY
############################

###### RUN 1
# Yes INNE, Cleanse, k_best_catboost
# param_grid = {
#     "model__C": [0.001, 0.1, 1, 10],
#     "model__penalty": ["l1", "l2", "elasticnet"],
#     "model__solver": ["saga"],
#     "model__l1_ratio": [0.5, 0, 1.0],
#     "model__max_iter": [500],
#     "model__class_weight": ["balanced", {1: 250}, {1: 1500}],
#     "selector__k": [25]
# }

In [10]:
# Optional: Add INNE anomaly features before training
# Hyperparameters default to the best values from standalone INNE tuning
# You can comment out this block if you want to test the model without INNE features
 
df = add_inne_features(
    df,

    # Default INNE parameters that were found to be best during standalone tuning.
    # You can adjust these if you want to test different INNE configurations.
    transaction_data_path="/dsa/groups/casestudycf25/team02/silver/cms_general_payments_anomaly_ready.csv",
    group_col="npi",
    year_col="year",
    txn_provider_col="covered_recipient_npi",
    txn_year_col="program_year",
    txn_target_col="target",
    txn_id_cols=["covered_recipient_npi", "record_id", "program_year"],
    n_estimators=100,
    max_samples=8,
    contamination=0.0005,
    top_k_features=15,
    null_fill=-1,
    verbose=1,
)

INNE: Loading /dsa/groups/casestudycf25/team02/silver/cms_general_payments_anomaly_ready.csv
INNE: 932908 transactions, 15 features selected
INNE: Fitted with n_estimators=100, max_samples=8, contamination=0.0005
INNE: Score range=[0.1916, 1.0000], P95 cutoff=0.9151
INNE: Rolled up to 158751 provider-years, 18285 with any above P95
INNE: Joined -> (817496, 159), 276190/817496 matched (33.8%)


In [11]:
pipe = MLPipeline(model_name="logistic_regression", target_col=target_col)

In [ ]:
# Check the split
print(f"Train: {len(pipe.y_train_)} samples, {pipe.y_train_.sum()} positive ({pipe.y_train_.mean():.4f})")
print(f"Test:  {len(pipe.y_test_)} samples, {pipe.y_test_.sum()} positive ({pipe.y_test_.mean():.4f})")

In [12]:
print(f"\n{'='*60}")
print(f"Running with feature_selection_method: k_best_catboost")
print(f"{'='*60}\n")

pipe.train(df, 
               param_grid=param_grid, 
               group_col="npi", 
               exclude_cols=exclude_cols, 
               undersample_method='RandomUnderSampler', 
               feature_selection_method="k_best_catboost")


Running with feature_selection_method: k_best_catboost


Undersampling (RandomUnderSampler) applied.
Class distribution after undersampling: {0: 2600, 1: 234}

Feature Selection (k_best_catboost) selected.
  Fold 1 validation prevalence (baseline AUPRC): 0.1034
  Fold 2 validation prevalence (baseline AUPRC): 0.0747
  Fold 3 validation prevalence (baseline AUPRC): 0.0796
  Fold 4 validation prevalence (baseline AUPRC): 0.0764
  Fold 5 validation prevalence (baseline AUPRC): 0.0780
Fitting 5 folds for each of 108 candidates, totalling 540 fits
[CV 2/5] END model__C=0.001, model__class_weight=balanced, model__l1_ratio=0.5, model__max_iter=500, model__penalty=l2, model__solver=saga, selector__k=25; auprc: (test=0.749) f1_macro: (test=0.743) precision_macro: (test=0.701) recall_macro: (test=0.829) roc_auc: (test=0.890) total time=   1.0s
[CV 2/5] END model__C=0.001, model__class_weight=balanced, model__l1_ratio=0.5, model__max_iter=500, model__penalty=elasticnet, model__solver=saga, selec

In [13]:
print(f"\n{'='*60}")
print(f"Running with feature_selection_method: k_best_xgb")
print(f"{'='*60}\n")
    
pipe.train(df, 
               param_grid=param_grid, 
               group_col="npi", 
               exclude_cols=exclude_cols, 
               undersample_method='RandomUnderSampler', 
               feature_selection_method="k_best_xgb")


Running with feature_selection_method: k_best_xgb


Undersampling (RandomUnderSampler) applied.
Class distribution after undersampling: {0: 2600, 1: 234}

Feature Selection (k_best_xgb) selected.
  Fold 1 validation prevalence (baseline AUPRC): 0.1034
  Fold 2 validation prevalence (baseline AUPRC): 0.0747
  Fold 3 validation prevalence (baseline AUPRC): 0.0796
  Fold 4 validation prevalence (baseline AUPRC): 0.0764
  Fold 5 validation prevalence (baseline AUPRC): 0.0780
Fitting 5 folds for each of 108 candidates, totalling 540 fits
[CV 4/5] END model__C=10, model__class_weight=balanced, model__l1_ratio=0, model__max_iter=500, model__penalty=l2, model__solver=saga, selector__k=25; auprc: (test=0.345) f1_macro: (test=0.549) precision_macro: (test=0.544) recall_macro: (test=0.581) roc_auc: (test=0.715) total time=   1.6s
[CV 1/5] END model__C=10, model__class_weight=balanced, model__l1_ratio=1.0, model__max_iter=500, model__penalty=l1, model__solver=saga, selector__k=25; auprc: (test=0.

In [14]:
print(f"\n{'='*60}")
print(f"Running with feature_selection_method: k_best_lgbm")
print(f"{'='*60}\n")
    
pipe.train(df, 
               param_grid=param_grid, 
               group_col="npi", 
               exclude_cols=exclude_cols, 
               undersample_method='RandomUnderSampler', 
               feature_selection_method="k_best_lgbm")


Running with feature_selection_method: k_best_lgbm


Undersampling (RandomUnderSampler) applied.
Class distribution after undersampling: {0: 2600, 1: 234}

Feature Selection (k_best_lgbm) selected.
  Fold 1 validation prevalence (baseline AUPRC): 0.1034
  Fold 2 validation prevalence (baseline AUPRC): 0.0747
  Fold 3 validation prevalence (baseline AUPRC): 0.0796
  Fold 4 validation prevalence (baseline AUPRC): 0.0764
  Fold 5 validation prevalence (baseline AUPRC): 0.0780
Fitting 5 folds for each of 108 candidates, totalling 540 fits
[CV 5/5] END model__C=1, model__class_weight={1: 250}, model__l1_ratio=0.5, model__max_iter=500, model__penalty=elasticnet, model__solver=saga, selector__k=25; auprc: (test=0.098) f1_macro: (test=0.407) precision_macro: (test=0.511) recall_macro: (test=0.539) roc_auc: (test=0.523) total time=   0.6s
[CV 3/5] END model__C=1, model__class_weight={1: 250}, model__l1_ratio=0, model__max_iter=500, model__penalty=l2, model__solver=saga, selector__k=25; auprc: 

In [15]:
print(f"\n{'='*60}")
print(f"Running with feature_selection_method: k_best_decision_tree")
print(f"{'='*60}\n")
    
pipe.train(df, 
               param_grid=param_grid, 
               group_col="npi", 
               exclude_cols=exclude_cols, 
               undersample_method='RandomUnderSampler', 
               feature_selection_method="k_best_decision_tree")


Running with feature_selection_method: k_best_decision_tree


Undersampling (RandomUnderSampler) applied.
Class distribution after undersampling: {0: 2600, 1: 234}

Feature Selection (k_best_decision_tree) selected.
  Fold 1 validation prevalence (baseline AUPRC): 0.1034
  Fold 2 validation prevalence (baseline AUPRC): 0.0747
  Fold 3 validation prevalence (baseline AUPRC): 0.0796
  Fold 4 validation prevalence (baseline AUPRC): 0.0764
  Fold 5 validation prevalence (baseline AUPRC): 0.0780
Fitting 5 folds for each of 108 candidates, totalling 540 fits
[CV 4/5] END model__C=1, model__class_weight={1: 1500}, model__l1_ratio=0.5, model__max_iter=500, model__penalty=l2, model__solver=saga, selector__k=25; auprc: (test=0.066) f1_macro: (test=0.380) precision_macro: (test=0.496) recall_macro: (test=0.486) roc_auc: (test=0.451) total time=  11.9s
[CV 1/5] END model__C=1, model__class_weight={1: 1500}, model__l1_ratio=0, model__max_iter=500, model__penalty=l1, model__solver=saga, selector__k

In [16]:
# for feature_selection_m in select_methods:
#     print(f"\n{'='*60}")
#     print(f"Running with feature_selection_method: {feature_selection_m}")
#     print(f"{'='*60}\n")
    
#     pipe.train(df, 
#                param_grid=param_grid, 
#                group_col="npi", 
#                exclude_cols=exclude_cols, 
#                undersample_method='CLEANSE', 
#                feature_selection_method=feature_selection_m)